# Extração e validação de notas fiscais eletrônicas (NFS-e / NF-e / CT-e)

Este notebook mostra, de forma didática, o núcleo do pipeline de extração e validação usado em um
sistema real de conferência automática de notas fiscais (NF-e/DANFE, NFS-e e CT-e) que desenvolvi.

**Importante:** todo o conteúdo aqui foi adaptado para fins educacionais/portfólio —
os exemplos de documentos são **sintéticos** (CNPJs, razões sociais e valores inventados),
sem nenhum dado real de cliente. A lógica de extração e as regras de validação, porém,
são as mesmas usadas em produção.

## O que o pipeline real faz

1. Recebe o PDF (ou XML) da nota fiscal.
2. Extrai o texto (com fallback para OCR quando o PDF não tem texto nativo).
3. Faz o parsing dos campos relevantes por regex (PDF) ou XPath (XML) — chave de acesso, CNPJs,
   valores, tributos, itens.
4. Valida os dados extraídos contra regras de negócio (RF03, RF05, RF06, RF07, RF09...) e contra
   APIs públicas (Receita Federal via Minha Receita, tabela IBPT/NCM via BrasilAPI).
5. Gera uma lista de **pendências** para revisão humana quando algo diverge do esperado.

Vamos percorrer cada uma dessas etapas.

## 1. Extração de texto do PDF

O primeiro desafio é conseguir um texto confiável a partir do PDF. Usamos `pdfplumber` com uma
tolerância de espaçamento (`x_tolerance`) baixa — sem isso, o texto do DANFSe (NFS-e do Padrão
Nacional) vem colado, sem espaço nenhum entre palavras, por causa do kerning apertado da fonte
usada pelo gerador oficial.

Quando o PDF não tem nenhuma camada de texto nativa (alguns portais municipais convertem a fonte
em contorno vetorial), caímos para OCR (Tesseract) sobre a imagem renderizada da página — e
sinalizamos isso explicitamente, porque OCR erra dígito com mais frequência que extração nativa.

In [1]:
import io
import re

import pdfplumber  # pip install pdfplumber

_LIMIAR_TEXTO_NATIVO = 20  # abaixo disso, tratamos o PDF como "sem texto nativo"


def extrair_texto_pdf(conteudo: bytes) -> tuple[str, bool]:
    """Extrai o texto de todas as páginas de um PDF.

    Retorna (texto, extraido_via_ocr). Quando o PDF não tem texto nativo,
    cai para OCR sobre a imagem renderizada da página.
    """
    partes = []
    with pdfplumber.open(io.BytesIO(conteudo)) as pdf:
        for pagina in pdf.pages:
            texto = pagina.extract_text(x_tolerance=1.5, y_tolerance=3) or ""
            partes.append(texto)
    texto = "\n".join(partes)
    if len(texto.strip()) >= _LIMIAR_TEXTO_NATIVO:
        return texto, False

    texto_ocr = _extrair_texto_via_ocr(conteudo)
    return (texto_ocr, True) if texto_ocr.strip() else (texto, False)


def _extrair_texto_via_ocr(conteudo: bytes) -> str:
    import pytesseract  # pip install pytesseract (+ tesseract-ocr no sistema)

    partes = []
    with pdfplumber.open(io.BytesIO(conteudo)) as pdf:
        for pagina in pdf.pages:
            imagem = pagina.to_image(resolution=300).original
            partes.append(pytesseract.image_to_string(imagem, lang="por"))
    return "\n".join(partes)


print("Função de extração definida. Nas próximas seções vamos trabalhar direto com o texto,")
print("já que o interesse aqui é o parsing — não a extração bruta do PDF.")

Função de extração definida. Nas próximas seções vamos trabalhar direto com o texto,
já que o interesse aqui é o parsing — não a extração bruta do PDF.


## 2. Parsing por regex — NFS-e (DANFSe v2.0, Padrão Nacional)

Depois de extrair o texto, o parser identifica cada campo com expressões regulares ancoradas nos
rótulos impressos pelo layout oficial da DANFSe. Como o layout nacional convive com layouts
municipais legados (ex.: Prefeitura de São Paulo), o parser tenta primeiro o layout nacional e só
recorre ao layout alternativo (`Layout B`) para os campos que ainda ficaram vazios.

Abaixo, uma versão simplificada do parser real rodando contra um texto **sintético**, no formato
que o `pdfplumber` produziria para uma DANFSe.

In [2]:
from datetime import datetime

CNPJ_CPF_RE = r"(\d{2}\.\d{3}\.\d{3}/\d{4}-\d{2}|\d{3}\.\d{3}\.\d{3}-\d{2})"


def brl_para_float(valor: str | None) -> float | None:
    """Converte um número em formato monetário brasileiro ("1.234,56") para float."""
    if valor is None:
        return None
    valor = valor.strip().replace(" ", "").replace("R$", "")
    if valor in ("", "-", "--"):
        return None
    valor = valor.replace(".", "").replace(",", ".")
    try:
        return float(valor)
    except ValueError:
        return None


def primeiro(padrao: str, texto: str, flags=re.IGNORECASE) -> str | None:
    m = re.search(padrao, texto, flags)
    return m.group(1).strip() if m else None


def _parse_data_hora(data, hora):
    if not data:
        return None
    fmt = "%d/%m/%Y %H:%M:%S" if hora else "%d/%m/%Y"
    try:
        return datetime.strptime(f"{data} {hora}".strip() if hora else data, fmt)
    except ValueError:
        return None


def parse_nfse(texto: str) -> dict:
    """Versão simplificada (para fins didáticos) do parser de NFS-e real —
    extrai os campos principais de uma DANFSe v2.0 a partir do texto do PDF."""
    chave_acesso = primeiro(r"CHAVE DE ACESSO DA NFS-e\s*\n(\d+)", texto)
    numero = primeiro(r"N[ÚU]MERO DA NFS-e\s+COMPET[ÊE]NCIA.*?\n(\d+)", texto, re.IGNORECASE | re.DOTALL)

    m_emissao = re.search(
        r"DATA E HORA DA EMISS[ÃA]O DA NFS-e\s*\n\d+\s+\d{2}/\d{2}/\d{4}\s+"
        r"(\d{2}/\d{2}/\d{4})\s+(\d{2}:\d{2}:\d{2})",
        texto, re.IGNORECASE,
    )
    data_emissao = _parse_data_hora(*m_emissao.groups()) if m_emissao else None

    cnpj_emitente = primeiro(r"PRESTADOR\s*/\s*FORNECEDOR.*?\n\s*" + CNPJ_CPF_RE, texto, re.IGNORECASE | re.DOTALL)
    cnpj_destinatario = primeiro(r"TOMADOR\s*/\s*ADQUIRENTE.*?\n\s*" + CNPJ_CPF_RE, texto, re.IGNORECASE | re.DOTALL)

    valor_total = brl_para_float(
        primeiro(r"VALOR TOTAL D[AE] NFS-e\s+VALOR DA OPERA[ÇC][ÃA]O.*?\n\s*R?\$?\s*([\d.,]+)", texto,
                 re.IGNORECASE | re.DOTALL)
    )

    codigo_tributacao = primeiro(
        r"C[óo]digo de Tributa[çc][ãa]o Nacional\s*/\s*Municipal.*?\n([\d.]+)", texto,
        re.IGNORECASE | re.DOTALL,
    )

    return {
        "chave_acesso": chave_acesso,
        "numero": numero,
        "data_emissao": data_emissao,
        "cnpj_emitente": cnpj_emitente,
        "cnpj_destinatario": cnpj_destinatario,
        "valor_total": valor_total,
        "codigo_tributacao_nacional": codigo_tributacao,
    }


# Texto sintético, no formato que o pdfplumber produz para uma DANFSe real
# (empresas e valores fictícios).
texto_exemplo = """DANFSe v2.0 Município: EXEMPLO - EX
CHAVE DE ACESSO DA NFS-e
12345678901234567890123456789012345678901234567890
NÚMERO DA NFS-e COMPETÊNCIA DA NFS-e DATA E HORA DA EMISSÃO DA NFS-e
1000123 01/09/2026 04/09/2026 14:22:10
PRESTADOR / FORNECEDOR CNPJ / CPF / NIF Indicador Municipal (Inscrição) Telefone
11.222.333/0001-44 12345 -
TOMADOR / ADQUIRENTE CNPJ / CPF / NIF Indicador Municipal (Inscrição) Telefone
99.888.777/0001-66 - -
Código de Tributação Nacional / Municipal Código da NBS Local da Prestação / Sigla UF / País
01.05.01 / 010 - EXEMPLO / EX / -
VALOR TOTAL DA NFS-e VALOR DA OPERAÇÃO VALOR APROXIMADO DOS TRIBUTOS
R$ 5.000,00 R$ 5.000,00 R$ 750,00
"""

documento = parse_nfse(texto_exemplo)
documento

{'chave_acesso': '12345678901234567890123456789012345678901234567890', 'numero': '1000123', 'data_emissao': datetime.datetime(2026, 9, 4, 14, 22, 10), 'cnpj_emitente': '11.222.333/0001-44', 'cnpj_destinatario': '99.888.777/0001-66', 'valor_total': 5000.0, 'codigo_tributacao_nacional': '01.05.01'}

## 3. Parsing via XML (Padrão Nacional NFSe/DPS)

Quando disponível, o XML é preferível ao PDF: os dados vêm estruturados, sem ambiguidade de
layout. O parser real usa XPath sobre o schema `NFSe/DPS v1.x`; abaixo, uma versão simplificada
rodando contra um XML sintético mínimo.

In [3]:
import xml.etree.ElementTree as ET

xml_exemplo = """<NFSe xmlns="http://www.sped.fazenda.gov.br/nfse">
  <infNFSe Id="NFS12345678901234567890123456789012345678901234">
    <nNFSe>1000123</nNFSe>
    <dhProc>2026-09-04T14:22:10-03:00</dhProc>
    <DPS>
      <infDPS>
        <prest><CNPJ>11222333000144</CNPJ><xNome>PRESTADOR EXEMPLO LTDA</xNome></prest>
        <toma><CNPJ>99888777000166</CNPJ><xNome>TOMADOR EXEMPLO LTDA</xNome></toma>
        <serv><cTribNac>010501</cTribNac></serv>
        <valores><vLiq>5000.00</vLiq><vBC>5000.00</vBC><pAliqAplic>5.00</pAliqAplic><vISSQN>250.00</vISSQN></valores>
      </infDPS>
    </DPS>
  </infNFSe>
</NFSe>"""


def _ns_strip(tag: str) -> str:
    return tag.split("}")[-1] if "}" in tag else tag


def find(elem, nome):
    if elem is None:
        return None
    for e in elem.iter():
        if _ns_strip(e.tag) == nome:
            return e
    return None


def campo(elem, nome):
    e = find(elem, nome)
    return e.text.strip() if e is not None and e.text else None


def numero(elem, nome):
    valor = campo(elem, nome)
    return float(valor) if valor is not None else None


def formatar_cnpj(cnpj: str | None) -> str | None:
    if not cnpj or len(cnpj) != 14:
        return cnpj
    return f"{cnpj[0:2]}.{cnpj[2:5]}.{cnpj[5:8]}/{cnpj[8:12]}-{cnpj[12:14]}"


def parse_nfse_xml(root) -> dict:
    inf_nfse = find(root, "infNFSe")
    prest = find(root, "prest")
    toma = find(root, "toma")
    serv = find(root, "serv")
    valores = find(root, "valores")

    return {
        "chave_acesso": inf_nfse.get("Id") if inf_nfse is not None else None,
        "numero": campo(inf_nfse, "nNFSe"),
        "cnpj_emitente": formatar_cnpj(campo(prest, "CNPJ")),
        "razao_social_emitente": campo(prest, "xNome"),
        "cnpj_destinatario": formatar_cnpj(campo(toma, "CNPJ")),
        "razao_social_destinatario": campo(toma, "xNome"),
        "codigo_tributacao_nacional": campo(serv, "cTribNac"),
        "valor_total": numero(valores, "vLiq"),
        "valor_issqn": numero(valores, "vISSQN"),
    }


root = ET.fromstring(xml_exemplo)
parse_nfse_xml(root)

{'chave_acesso': 'NFS12345678901234567890123456789012345678901234', 'numero': '1000123', 'cnpj_emitente': '11.222.333/0001-44', 'razao_social_emitente': 'PRESTADOR EXEMPLO LTDA', 'cnpj_destinatario': '99.888.777/0001-66', 'razao_social_destinatario': 'TOMADOR EXEMPLO LTDA', 'codigo_tributacao_nacional': '010501', 'valor_total': 5000.0, 'valor_issqn': 250.0}

## 4. Validação de CNPJ/CPF e chave de acesso

Antes de confiar em qualquer campo extraído, validamos os identificadores com os algoritmos
oficiais de dígito verificador: módulo 11 para CNPJ/CPF, e o mesmo módulo 11 (pesos cíclicos
2-9) usado na chave de acesso de 44 dígitos de NF-e/CT-e. A NFS-e do Padrão Nacional usa uma
chave de 50 dígitos com algoritmo de DV próprio — validamos aqui só o tamanho.

In [4]:
def _somente_digitos(valor):
    return re.sub(r"\D", "", valor or "")


def validar_cnpj(cnpj: str | None) -> bool:
    digitos = _somente_digitos(cnpj)
    if len(digitos) != 14 or digitos == digitos[0] * 14:
        return False

    def dv(base, pesos):
        soma = sum(int(d) * p for d, p in zip(base, pesos))
        resto = soma % 11
        return 0 if resto < 2 else 11 - resto

    dv1 = dv(digitos[:12], [5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2])
    dv2 = dv(digitos[:12] + str(dv1), [6, 5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2])
    return digitos[-2:] == f"{dv1}{dv2}"


def _dv_chave_44(chave_43_digitos: str) -> int:
    """Mesmo algoritmo (mod 11, pesos 2-9 cíclicos da direita p/ esquerda) usado pela SEFAZ."""
    peso, soma = 2, 0
    for c in reversed(chave_43_digitos):
        soma += int(c) * peso
        peso = 2 if peso == 9 else peso + 1
    resto = soma % 11
    return 0 if resto < 2 else 11 - resto


def validar_chave_acesso(chave: str | None, tipo: str) -> bool:
    digitos = _somente_digitos(chave)
    if tipo in ("nfe", "cte"):
        return len(digitos) == 44 and _dv_chave_44(digitos[:43]) == int(digitos[43])
    if tipo == "nfse":
        return len(digitos) == 50
    return False


# Exemplos (CNPJ gerado apenas para teste do algoritmo, não corresponde a empresa real)
print("CNPJ válido (11.222.333/0001-81):", validar_cnpj("11.222.333/0001-81"))
print("CNPJ inválido (11.222.333/0001-00):", validar_cnpj("11.222.333/0001-00"))
print("Chave NFS-e (50 dígitos):", validar_chave_acesso(documento["chave_acesso"], "nfse"))

CNPJ válido (11.222.333/0001-81): True
CNPJ inválido (11.222.333/0001-00): False
Chave NFS-e (50 dígitos): True


## 5. Consumindo APIs públicas para enriquecer e conferir a nota

Duas integrações fazem a ponte entre o documento e a realidade:

- **[Minha Receita](https://minhareceita.org)** — espelha os dados públicos de CNPJ da Receita
  Federal. Usamos para conferir a situação cadastral do emitente e o regime de tributação
  (Simples Nacional) declarado.
- **[BrasilAPI](https://brasilapi.com.br)** — tabela oficial de NCM (Camex) e tabela IBPT
  (Lei 12.741/2012) por UF, usada para recalcular o valor aproximado de tributos.

Ambas são públicas, sem autenticação. No sistema real, toda chamada é *best-effort*: se a rede
falhar, isso vira "não verificado" — nunca "confirmadamente inválido" (uma API fora do ar não pode
virar pendência falsa em toda nota).

As células abaixo fazem chamadas de verdade, usando o CNPJ público de uma empresa de capital
aberto (dado público de registro, não é informação sensível) só para ilustrar o formato da
resposta.

In [5]:
import httpx


def consultar_cnpj(cnpj: str) -> dict | None:
    digitos = _somente_digitos(cnpj)
    resp = httpx.get(f"https://minhareceita.org/{digitos}", timeout=8.0)
    return resp.json() if resp.status_code == 200 else None


# CNPJ matriz do Banco do Brasil — dado público de registro na Receita Federal.
dados_cnpj = consultar_cnpj("00.000.000/0001-91")
if dados_cnpj:
    print("Razão social:", dados_cnpj.get("razao_social"))
    print("Situação cadastral:", dados_cnpj.get("descricao_situacao_cadastral"))
    print("Opção pelo Simples Nacional:", dados_cnpj.get("opcao_pelo_simples"))
else:
    print("API indisponível no momento — no sistema real isso vira 'não verificado', nunca 'inválido'.")

Razão social: BANCO DO BRASIL SA
Situação cadastral: ATIVA
Opção pelo Simples Nacional: False


In [6]:
def consultar_ncm(codigo: str) -> dict | None:
    resp = httpx.get(f"https://brasilapi.com.br/api/ncm/v1/{codigo}", timeout=5.0)
    if resp.status_code == 404:
        return None
    resp.raise_for_status()
    return resp.json()


# NCM de notebooks/laptops — tabela pública da Camex.
ncm = consultar_ncm("8471.30.12")
ncm

{'codigo': '8471.30.12', 'descricao': 'De peso inferior a 3,5\xa0kg, com tela de área superior a 140\xa0cm<sup>2</sup>, mas inferior a 560\xa0cm<sup>2</sup>', 'data_inicio': '2022-04-01', 'data_fim': '9999-12-31', 'tipo_ato': 'Res Camex', 'numero_ato': '000272', 'ano_ato': '2021'}

## 6. Recalculando o valor aproximado de tributos (Lei 12.741/2012)

A Lei da Transparência Fiscal exige que a nota informe o valor aproximado de tributos federais,
estaduais e municipais embutidos no preço. Conferimos esse valor recalculando pela tabela IBPT
(alíquota aproximada por NCM/UF), com uma tolerância generosa de 20% — a lei já trata o valor como
"aproximado", e a tabela pode estar desatualizada.

In [7]:
def obter_aliquota_ibpt_media(uf: str, codigo_ncm: str) -> dict | None:
    """Busca a alíquota IBPT para um NCM específico numa UF (best-effort)."""
    resp = httpx.get(f"https://brasilapi.com.br/api/ibpt/ncm/v1/{uf.upper()}", timeout=20.0)
    if resp.status_code != 200:
        return None
    for linha in resp.json():
        if (linha.get("codigo") or "").strip() == codigo_ncm:
            return {
                "federal": linha.get("nacional_federal") or 0,
                "estadual": linha.get("estadual") or 0,
                "municipal": linha.get("municipal") or 0,
            }
    return None


_TOLERANCIA_RELATIVA_IBPT = 0.20

# Item sintético: um notebook vendido por R$ 5.000,00 em SP, com um valor
# aproximado de tributos declarado (fictício) de R$ 750,00 na nota.
valor_item = 5000.00
valor_aprox_declarado = 750.00

aliquota = obter_aliquota_ibpt_media("SP", "84713012")
if aliquota:
    percentual = (aliquota["federal"] + aliquota["estadual"] + aliquota["municipal"]) / 100
    valor_esperado = valor_item * percentual
    diferenca_relativa = abs(valor_esperado - valor_aprox_declarado) / valor_esperado if valor_esperado else None

    print(f"Alíquota IBPT (federal+estadual+municipal): {percentual:.2%}")
    print(f"Valor esperado pela tabela IBPT: R$ {valor_esperado:.2f}")
    print(f"Valor declarado na nota:        R$ {valor_aprox_declarado:.2f}")
    if diferenca_relativa is not None:
        status = "DIVERGE" if diferenca_relativa > _TOLERANCIA_RELATIVA_IBPT else "OK"
        print(f"Diferença relativa: {diferenca_relativa:.1%} -> {status}")
else:
    print("NCM não encontrado na tabela IBPT dessa UF (ou API indisponível).")

Alíquota IBPT (federal+estadual+municipal): 33.15%
Valor esperado pela tabela IBPT: R$ 1657.50
Valor declarado na nota:        R$ 750.00
Diferença relativa: 54.8% -> DIVERGE


## 7. A cereja do bolo: checagem semântica com a Gemini API

Nem toda divergência é uma conta exata. O código de tributação declarado (ex.: "20.01.01 —
Serviços portuários") é uma categoria fixa e genérica; a descrição livre do serviço, escrita pelo
prestador, é sempre mais específica. Comparar as duas por regra determinística geraria falso
positivo toda hora.

Para esse caso específico, o sistema real faz **uma única pergunta objetiva a um LLM** (Gemini,
free tier): "essa descrição livre é plausivelmente compatível com essa categoria oficial?". É a
única parte do pipeline que usa IA generativa — e, por ser uma estimativa de significado (não uma
conta exata como CFOP/UF), a pendência gerada é **sempre de severidade baixa**, nunca bloqueia a
nota.

Reparem no padrão: a chamada é **best-effort** — se a `GEMINI_API_KEY` não estiver configurada, ou
a API falhar, a checagem é simplesmente pulada. Isso nunca pode travar a validação nem virar "erro
confirmado" só porque uma dependência externa está fora do ar.

In [8]:
import os

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/models"
GEMINI_MODELO = "gemini-flash-latest"


class ConsultaIndisponivel(Exception):
    pass


def comparar_servico_com_categoria(descricao_servico: str, categoria_oficial: str) -> dict | None:
    """Pergunta ao modelo se a descrição livre do serviço é compatível com a
    categoria oficial de tributação declarada. Best-effort: nunca deve travar
    a validação — se algo falhar, quem chama trata como "não verificado"."""
    chave = os.environ.get("GEMINI_API_KEY")
    if not chave:
        raise ConsultaIndisponivel("GEMINI_API_KEY não configurada — checagem semântica pulada.")

    prompt = (
        "Você confere notas fiscais de serviço (NFS-e) no Brasil. Vou te dar duas informações "
        "do mesmo documento: (1) a categoria oficial de tributação, tirada da lista nacional de "
        "serviços; (2) a descrição livre do serviço que o prestador escreveu, específica daquela "
        "operação. Diga se a descrição livre é compatível com a categoria oficial. Considere que "
        "a descrição livre é sempre mais específica que a categoria (isso é esperado, não é "
        "divergência). Só aponte incompatibilidade se o serviço descrito for de um ramo "
        "claramente diferente da categoria.\\n\\n"
        f"Categoria oficial: {categoria_oficial}\\n\\n"
        f"Descrição do serviço: {descricao_servico}\\n\\n"
        'Responda SOMENTE com um JSON no formato exato: {"compativel": true ou false, '
        '"justificativa": "uma frase curta explicando por quê"}'
    )

    resp = httpx.post(
        f"{GEMINI_BASE_URL}/{GEMINI_MODELO}:generateContent",
        headers={"Content-Type": "application/json", "X-goog-api-key": chave},
        json={"contents": [{"parts": [{"text": prompt}]}]},
        timeout=15.0,
    )
    if resp.status_code != 200:
        raise ConsultaIndisponivel(f"Gemini API respondeu {resp.status_code}")

    import json as _json
    texto_resposta = resp.json()["candidates"][0]["content"]["parts"][0]["text"].strip()
    if texto_resposta.startswith("```"):
        texto_resposta = texto_resposta.strip("`").removeprefix("json").strip()
    dados = _json.loads(texto_resposta)
    return {"compativel": bool(dados["compativel"]), "justificativa": str(dados.get("justificativa", ""))}


# Categoria oficial (do código de tributação) x descrição livre extraídas na
# seção 2 — ambas fictícias, mas no formato real de uma DANFSe.
categoria_oficial = "01.05.01 — Assessoria ou consultoria de qualquer natureza"
descricao_servico = "Consultoria de implantação de ERP financeiro, incluindo parametrização de módulo fiscal"

try:
    verificacao = comparar_servico_com_categoria(descricao_servico, categoria_oficial)
    print("Compatível:", verificacao["compativel"])
    print("Justificativa:", verificacao["justificativa"])
except ConsultaIndisponivel as exc:
    # Este é o caminho que roda de fato aqui no notebook público, já que
    # nenhuma GEMINI_API_KEY está configurada neste ambiente — e é
    # exatamente esse o comportamento que o sistema real precisa garantir:
    # a ausência da IA generativa NUNCA impede a nota de ser validada.
    verificacao = None
    print(f"Checagem semântica pulada: {exc}")
    print("-> a nota segue sendo validada normalmente pelas outras regras (RF03, RF05, RF06...).")

Checagem semântica pulada: GEMINI_API_KEY não configurada — checagem semântica pulada.
-> a nota segue sendo validada normalmente pelas outras regras (RF03, RF05, RF06...).


## 8. Regras de negócio → pendências

A última etapa cruza tudo isso: campos obrigatórios ausentes, CNPJ/chave inválidos, situação
cadastral irregular na Receita, e as regras específicas de cada tipo de documento (ex.: CFOP
incompatível com o sentido da operação num DANFE, retenção de ISSQN indevida numa NFS-e cujo
código de serviço está na lista de "sem retenção obrigatória", IBS/CBS zerado). Cada regra tem um
identificador rastreável (RF03, RF05, RF06, RF07, RF09, RN02, RN03...) e vira uma pendência
estruturada para revisão humana — a validação nunca decide sozinha que uma nota está errada.

In [9]:
def pend(regra, tipo, descricao, item_index=None):
    return {"regra_aplicada": regra, "tipo": tipo, "descricao": descricao, "item_index": item_index}


def validar_nfse(
    documento: dict, *, dados_cnpj_emitente: dict | None = None, verificacao_semantica: dict | None = None
) -> list[dict]:
    pendencias = []

    if not validar_chave_acesso(documento.get("chave_acesso"), "nfse"):
        pendencias.append(pend("RF06", "chave_acesso_invalida",
                                f"Chave de acesso ausente ou com tamanho incorreto: {documento.get('chave_acesso')!r}."))

    if not validar_cnpj(documento.get("cnpj_emitente")):
        pendencias.append(pend("RF05", "cnpj_emitente_invalido",
                                f"CNPJ do prestador inválido: {documento.get('cnpj_emitente')!r}."))

    if dados_cnpj_emitente is not None:
        situacao = dados_cnpj_emitente.get("descricao_situacao_cadastral")
        if situacao and situacao.strip().upper() != "ATIVA":
            pendencias.append(pend("RF03", "emitente_situacao_cadastral_irregular",
                                    f"Situação cadastral do prestador na Receita Federal: {situacao} (não ATIVA)."))

    for campo_, rotulo in {"numero": "número", "valor_total": "valor total"}.items():
        if not documento.get(campo_):
            pendencias.append(pend("RF03", f"{campo_}_ausente", f'Campo obrigatório "{rotulo}" não encontrado.'))

    # RF08 — sempre baixa severidade: é uma estimativa de significado (via
    # LLM), não uma conta exata. Só entra em jogo quando a checagem rodou.
    if verificacao_semantica is not None and verificacao_semantica.get("compativel") is False:
        pendencias.append(pend("RF08", "servico_pode_nao_bater_com_codigo",
                                "O serviço descrito pode não corresponder à categoria oficial declarada — "
                                f"conferir manualmente. {verificacao_semantica.get('justificativa', '')}".strip()))

    return pendencias


# Reaproveitando o documento sintético extraído na seção 2, o resultado (real,
# se a API respondeu) da consulta de CNPJ da seção 5, e o resultado (real ou
# pulado) da checagem semântica via Gemini da seção 7.
# O CNPJ do prestador no texto de exemplo foi inventado sem dígito verificador
# válido — de propósito, para mostrar a validação pegando o problema de verdade.
pendencias = validar_nfse(documento, dados_cnpj_emitente=dados_cnpj, verificacao_semantica=verificacao)
pendencias or "Nenhuma pendência encontrada."

[{'regra_aplicada': 'RF05', 'tipo': 'cnpj_emitente_invalido', 'descricao': "CNPJ do prestador inválido: '11.222.333/0001-44'.", 'item_index': None}]

## Encerramento

Esse é o núcleo do pipeline: **extrair → estruturar → validar contra regras e fontes públicas →
gerar pendências rastreáveis para revisão humana**. Em produção, isso roda dentro de uma API que
recebe o PDF/XML, persiste o documento e as pendências, e notifica quem precisa revisar.

Feito por Yago Cavalcante como parte de um sistema de conferência automática de notas fiscais.
Dúvidas ou sugestões, fique à vontade para abrir uma issue.